# 🔬 Solutions: PDB Structure Analysis

Complete solutions for Kodomo Exercise 02

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This solution notebook demonstrates one reliable implementation path and can be used as a reference workflow.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Focus on why each step exists, not only the final answer.
- Compare intermediate outputs to your own attempt, not just final metrics.
- Small implementation differences can still be correct if assumptions match.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
import numpy as np
import math
from collections import defaultdict

## Exercise 1: Parse Cα Atoms

In [ ]:
def parse_pdb_ca_atoms(pdb_file):
    """Extract Cα atom coordinates from PDB file."""
    ca_atoms = []
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM  ') and line[12:16] == ' CA ':
                ca_atoms.append({
                    'residue': line[17:20].strip(),
                    'chain': line[21],
                    'res_num': int(line[22:26].strip()),
                    'x': float(line[30:38]),
                    'y': float(line[38:46]),
                    'z': float(line[46:54])
                })
    return ca_atoms

## Exercise 2: Calculate Center of Mass

In [ ]:
ATOMIC_MASSES = {'C': 12.011, 'N': 14.007, 'O': 15.999, 'S': 32.065, 'H': 1.008}

def parse_all_atoms(pdb_file):
    """Parse all ATOM records from PDB."""
    atoms = []
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM  '):
                atoms.append({
                    'element': line[76:78].strip() or line[12:14].strip()[0],
                    'x': float(line[30:38]),
                    'y': float(line[38:46]),
                    'z': float(line[46:54])
                })
    return atoms

def calculate_center_of_mass(atoms):
    """Calculate center of mass from atom coordinates."""
    total_mass = 0
    weighted_coords = np.array([0.0, 0.0, 0.0])
    
    for atom in atoms:
        mass = ATOMIC_MASSES.get(atom['element'].upper(), 12.0)
        coords = np.array([atom['x'], atom['y'], atom['z']])
        weighted_coords += mass * coords
        total_mass += mass
    
    return weighted_coords / total_mass if total_mass > 0 else weighted_coords

## Exercise 3: Count Chains and Residues

In [ ]:
def analyze_pdb_structure(pdb_file):
    """Analyze PDB structure: chains, residues, atoms."""
    chains = set()
    residues = set()
    atom_count = 0
    
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM  '):
                chains.add(line[21])
                residues.add((line[21], line[22:26].strip()))
                atom_count += 1
    
    return {
        'chains': sorted(chains),
        'num_chains': len(chains),
        'num_residues': len(residues),
        'num_atoms': atom_count
    }

## Exercise 4: Calculate Distances

In [ ]:
def euclidean_distance(atom1, atom2):
    """Calculate Euclidean distance between two atoms."""
    return math.sqrt(
        (atom1['x'] - atom2['x'])**2 +
        (atom1['y'] - atom2['y'])**2 +
        (atom1['z'] - atom2['z'])**2
    )

def ca_distance_matrix(ca_atoms):
    """Calculate distance matrix between all Cα atoms."""
    n = len(ca_atoms)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            dist = euclidean_distance(ca_atoms[i], ca_atoms[j])
            matrix[i][j] = matrix[j][i] = dist
    return matrix

def find_contacts(ca_atoms, cutoff=8.0):
    """Find residue contacts within cutoff distance."""
    contacts = []
    for i, a1 in enumerate(ca_atoms):
        for j, a2 in enumerate(ca_atoms[i+1:], i+1):
            dist = euclidean_distance(a1, a2)
            if dist < cutoff:
                contacts.append((a1['res_num'], a2['res_num'], dist))
    return contacts

## Exercise 5: B-factor Analysis

In [ ]:
def parse_bfactors(pdb_file):
    """Extract B-factors from PDB file."""
    bfactors = []
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM  '):
                bfactors.append({
                    'res_num': int(line[22:26].strip()),
                    'bfactor': float(line[60:66])
                })
    return bfactors

def average_bfactor_per_residue(bfactors):
    """Calculate average B-factor per residue."""
    residue_bfactors = defaultdict(list)
    for bf in bfactors:
        residue_bfactors[bf['res_num']].append(bf['bfactor'])
    return {res: np.mean(vals) for res, vals in residue_bfactors.items()}

def find_flexible_regions(avg_bfactors, threshold=50.0):
    """Find flexible regions (high B-factor)."""
    return [res for res, bf in avg_bfactors.items() if bf > threshold]